# ✈️ Havalimanı FOD (Foreign Object Debris) Tespiti — Ultralytics YOLO

Bu notebook baştan sona şunları yapar:

| Aşama | Ne yapılır |
|---|---|
| 1 | Ortam, GPU, güvenli (`pathlib`) yol yönetimi |
| 2 | Etiket doğrulama + sınıf/boyut istatistikleri |
| 3 | **Veri sızıntısı denetimi** (dosya adı, MD5, Roboflow kaynak kimliği, görsel benzerlik hash'i) ve sızıntılı train görsellerinin otomatik dışlanması |
| 4 | **Sınıf dengesizliği**: Repeat-Factor Sampling + sınıf ağırlıklı focal BCE (özel kayıp) + güçlü augmentation |
| 5 | **Overfitting önlemi**: early stopping, AdamW + weight decay, cosine LR, `close_mosaic`, EMA |
| 6 | Eğitim (yalnızca `train` + `valid`; `test` **fiziksel olarak** eğitim yaml'ında yok) |
| 7 | Eşik seçimi `valid` üzerinde, nihai bağımsız değerlendirme `test` üzerinde (Confusion Matrix, P/R/F1) |
| 8 | Çıkarım + kutu çizimi, ONNX export, özet JSON |

**Google Colab için hazır:** `Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU` seçin ve GPU olarak **G4** işaretleyin (96 GB VRAM; bu notebook'ta batch/cache buna göre ayarlı). Hücreleri yukarıdan aşağıya çalıştırın. Tüm ayarlar *Konfigürasyon* hücresindedir.

## 1) Kurulum

In [ ]:
# Ultralytics'in en güncel kararlı sürümünü kurar/günceller (Colab'da PyTorch+CUDA hazır gelir).
%pip install -q -U ultralytics pyyaml pandas matplotlib pillow tqdm

In [ ]:
import os, sys, json, math, random, shutil, hashlib, platform, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import ultralytics
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

if torch.cuda.is_available():
    DEVICE = 0
    print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    print("⚠️ GPU bulunamadı, CPU ile eğitim çok yavaş olacaktır.")

if DEVICE == 0:
    cc = torch.cuda.get_device_capability(0)
    tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    print("CUDA compute capability:", cc)
    if cc[0] >= 12 and tv < (2, 7):
        print("⚠️ Bu GPU (Blackwell) için PyTorch >= 2.7 (CUDA 12.8) gerekir. Kurun: %pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128 ve Runtime > Restart session yapın.")

print(f"Python {platform.python_version()} | torch {torch.__version__} | ultralytics {ultralytics.__version__} | device={DEVICE}")

### Google Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount("/content/drive")   # açılan pencerede izin verin
# Colab'da Drive yolu her zaman /content/drive/MyDrive'dır (Türkçe arayüzde 'Drive'ım' olarak görünse bile).

## 2) Konfigürasyon

Tüm ayarlar burada. Başka hiçbir hücrede elle değişiklik gerekmez.

**Önemli tasarım kararları**
- Veri seti **My Drive > datasets > FOD_dataset_robofllow** içinde olmalı (`/content/drive/MyDrive/datasets/FOD_dataset_robofllow`). Farklı bir yerdeyse `DATASET_ROOT`'u düzeltin.
- Drive bağlantısı (FUSE) küçük dosyalarda çok yavaştır; bu yüzden `USE_LOCAL_COPY = True` ile veri seti Colab'ın yerel diskine (`/content/fod_local`) kopyalanır. `data.yaml` ve liste dosyaları da orada üretilir; Drive'daki orijinal verinize dokunulmaz.
- Eğitim çıktıları (`last.pt`, `best.pt`, grafikler) **Drive'a** yazılır; Colab oturumu kopsa bile kaybolmaz ve `RESUME` ile devam edebilirsiniz.

In [ ]:
# ============================ VERİ SETİ ============================
DATASET_ROOT   = Path("/content/drive/MyDrive/datasets/FOD_dataset_robofllow")
USE_LOCAL_COPY = True    # Drive yavaş: veri setini /content altına kopyalayıp oradan çalış (önerilir)
LOCAL_COPY_DIR = Path("/content/fod_local")
RUNS_DIR       = Path("/content/drive/MyDrive/runs_fod")   # çıktılar Drive'a yazılır (oturum kopsa da kalır)
RUN_NAME       = "fod_yolo"

# ============================ SIZINTI DENETİMİ =====================
EXCLUDE_LEAKS       = False  # False: sızıntıyı yalnızca RAPORLA, hiçbir görseli eğitimden/valid'den çıkarma
RUN_NEAR_DUP        = True   # görsel benzerlik (dHash) ile near-duplicate taraması
NEAR_DUP_HAMMING    = 2      # 64-bit dHash'te en fazla bu kadar bit farkı = "aynı görsel" say

# ============================ SINIF DENGESİZLİĞİ ===================
USE_RFS             = True   # Repeat-Factor Sampling: nadir sınıf içeren train görsellerini tekrarla
RFS_T               = 0.10   # eşik: görsel-frekansı bundan düşük sınıflar tekrar alır
RFS_MAX_REPEAT      = 4      # bir görselin en fazla kaç kez tekrarlanacağı (overfit'i sınırlar)
USE_WEIGHTED_FOCAL  = True   # sınıf ağırlıklı focal-BCE (özel kayıp)
FOCAL_GAMMA         = 1.0    # 0 = focal yok, sadece sınıf ağırlığı
CLASS_W_POWER       = 0.5    # ağırlık = (medyan/sayı)^power  (0.5 = karekök, yumuşak)
CLASS_W_CLIP        = (0.5, 3.0)

# ============================ MODEL / EĞİTİM =======================
# Sırayla denenir; ilki indirilemez/tanınmazsa bir sonrakine düşer.
MODEL_CANDIDATES    = ["yolo26m.pt", "yolo11m.pt"]
IMGSZ               = 960    # FOD nesneleri küçüktür; 960 küçük nesne recall'ünü artırır (VRAM yetmezse 800/640)
EPOCHS              = 150
PATIENCE            = 30     # fitness (0.1*mAP50 + 0.9*mAP50-95, valid) bu kadar epoch iyileşmezse dur
BATCH               = 32 if DEVICE == 0 else 8     # G4 (96 GB) için 32 yeterli; AutoBatch aşırı büyük batch seçip iterasyon sayısını düşürür. -1 = AutoBatch
WORKERS             = min(8, os.cpu_count() or 2)   # Colab genelde 2-8 CPU
OPTIMIZER           = "AdamW"
LR0, LRF            = 1e-3, 0.01
WEIGHT_DECAY        = 1e-3
WARMUP_EPOCHS       = 3.0
CLOSE_MOSAIC        = 15     # son N epoch mosaic kapanır (gerçek dağılıma yakınsama)
FREEZE              = 0      # >0: ilk N katmanı dondur (çok küçük veri setinde overfit'i azaltabilir)
CACHE               = "auto" # "auto": RAM yetiyorsa "ram", yetmiyorsa kapalı | "ram" | False
RESUME              = False  # True: son eğitimi kaldığı yerden sürdür (aşağıdaki RESUME_WEIGHTS ile)
RESUME_WEIGHTS      = None   # oturum koptuysa örn. RUNS_DIR/"fod_yolo"/"weights"/"last.pt" (önce 1-3. bölümleri tekrar çalıştırın)

# ============================ DEĞERLENDİRME ========================
VAL_BATCH           = 16
EXPORT_ONNX         = True

## 3) Dizin yapısı doğrulama

`pathlib` ile yollar güvenle birleştirilir; `valid` yerine `val` klasörü varsa otomatik algılanır.

In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Veri seti kök dizini bulunamadı: {DATASET_ROOT}\n"
        f"Üst dizin mevcut mu: {DATASET_ROOT.parent.exists()}  -> Drive bağlama hücresini çalıştırdınız mı ve klasör yolu doğru mu (My Drive > datasets > FOD_dataset_robofllow)?"
    )

ROOT = DATASET_ROOT
if USE_LOCAL_COPY:
    if LOCAL_COPY_DIR.exists():
        shutil.rmtree(LOCAL_COPY_DIR)
    print(f"Kopyalanıyor: {DATASET_ROOT} -> {LOCAL_COPY_DIR}")
    shutil.copytree(DATASET_ROOT, LOCAL_COPY_DIR, ignore=shutil.ignore_patterns("*.cache"))
    ROOT = LOCAL_COPY_DIR
ROOT = ROOT.resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS  = ("train", "valid", "test")
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

SPLIT_DIR = {}
for s in SPLITS:
    candidates = [s] + (["val"] if s == "valid" else [])
    found = next((c for c in candidates if (ROOT / c / "images").is_dir() and (ROOT / c / "labels").is_dir()), None)
    if found is None:
        raise FileNotFoundError(f"'{s}' için images/labels klasörleri bulunamadı: {ROOT / s}")
    SPLIT_DIR[s] = found

IMAGES = {s: sorted(p for p in (ROOT / SPLIT_DIR[s] / "images").iterdir() if p.suffix.lower() in IMG_EXT) for s in SPLITS}

print("Kök dizin  :", ROOT)
print("ASCII yol  :", str(ROOT).isascii())
for s in SPLITS:
    print(f"{s:6s} -> klasör='{SPLIT_DIR[s]}' | görsel sayısı={len(IMAGES[s])}")
    if len(IMAGES[s]) == 0:
        raise RuntimeError(f"{s} kümesinde görsel yok!")

non_ascii_files = [p for s in SPLITS for p in IMAGES[s] if not p.name.isascii()]
if non_ascii_files:
    print(f"⚠️ {len(non_ascii_files)} dosya adı ASCII değil (örn. {non_ascii_files[0].name}). Liste dosyalarında sorun çıkarsa USE_LOCAL_COPY=True yapın.")

## 4) Sınıf adları + etiket doğrulama + istatistikler

- Mevcut Roboflow `data.yaml` içindeki `names` okunur (yoksa etiketlerden `class_i` üretilir).
- Her etiket satırı doğrulanır: 5 sütun, tam sayı sınıf id, normalize koordinat aralığı.
- Sınıf id'si `names` aralığı dışındaysa eğitim **durdurulur** (sessiz hatayı önlemek için).

In [ ]:
# ---- sınıf adları
NAMES = None
orig_yaml = ROOT / "data.yaml"
if orig_yaml.exists():
    try:
        d = yaml.safe_load(orig_yaml.read_text(encoding="utf-8"))
        n = d.get("names")
        if isinstance(n, dict):
            NAMES = [str(n[k]) for k in sorted(n, key=lambda x: int(x))]
        elif isinstance(n, list):
            NAMES = [str(x) for x in n]
    except Exception as e:
        print("Mevcut data.yaml okunamadı:", e)

# ---- etiketleri ayrıştır
RECORDS, ISSUES = {s: [] for s in SPLITS}, []
for s in SPLITS:
    lab_dir = ROOT / SPLIT_DIR[s] / "labels"
    for img in tqdm(IMAGES[s], desc=f"etiketler/{s}"):
        lab = lab_dir / (img.stem + ".txt")
        cls_list, boxes = [], []
        if lab.exists():
            for ln, line in enumerate(lab.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
                parts = line.split()
                if not parts:
                    continue
                if len(parts) != 5:
                    ISSUES.append((s, lab.name, ln, "5 sütun değil (segment/bozuk satır)")); continue
                try:
                    c = float(parts[0]); x, y, w, h = map(float, parts[1:])
                except ValueError:
                    ISSUES.append((s, lab.name, ln, "sayısal olmayan değer")); continue
                if c < 0 or c != int(c):
                    ISSUES.append((s, lab.name, ln, f"geçersiz sınıf id: {c}")); continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    ISSUES.append((s, lab.name, ln, "koordinat [0,1] dışında")); continue
                cls_list.append(int(c)); boxes.append((x, y, w, h))
        else:
            ISSUES.append((s, img.name, 0, "etiket dosyası yok (arka plan görseli sayılacak)"))
        RECORDS[s].append({"img": img, "cls": cls_list, "boxes": boxes})

# etiket var, görsel yok
for s in SPLITS:
    stems = {p.stem for p in IMAGES[s]}
    for lab in (ROOT / SPLIT_DIR[s] / "labels").glob("*.txt"):
        if lab.stem not in stems:
            ISSUES.append((s, lab.name, 0, "görseli olmayan etiket (yetim)"))

max_id = max((c for s in SPLITS for r in RECORDS[s] for c in r["cls"]), default=-1)
if NAMES is None:
    NAMES = [f"class_{i}" for i in range(max_id + 1)]
    print("⚠️ data.yaml'da names yok; etiketlerden üretildi:", NAMES)
if max_id >= len(NAMES):
    raise ValueError(f"Etiketlerde sınıf id={max_id} var ama names yalnızca {len(NAMES)} sınıf içeriyor. data.yaml'ı kontrol edin.")
NC = len(NAMES)

print(f"\nSınıf sayısı (nc) = {NC}: {NAMES}")
iss = pd.DataFrame(ISSUES, columns=["küme", "dosya", "satır", "sorun"])
if len(iss):
    print(f"\n⚠️ {len(iss)} etiket uyarısı. Sorun türüne göre özet:")
    display(iss.groupby(["küme", "sorun"]).size().rename("adet").reset_index())
    bad = iss[~iss["sorun"].str.contains("arka plan")]
    if len(bad):
        print("Bozuk satırlar Ultralytics tarafından atlanır; yine de inceleyin. İlk 10:")
        display(bad.head(10))
else:
    print("✅ Etiketlerde sorun bulunmadı.")

In [ ]:
# ---- sınıf dağılımı (instance ve görsel bazında)
inst = pd.DataFrame({s: pd.Series(Counter(c for r in RECORDS[s] for c in r["cls"]), dtype=int) for s in SPLITS}).reindex(range(NC)).fillna(0).astype(int)
imgs_with = pd.DataFrame({s: pd.Series(Counter(c for r in RECORDS[s] for c in set(r["cls"])), dtype=int) for s in SPLITS}).reindex(range(NC)).fillna(0).astype(int)
inst.index = imgs_with.index = NAMES

print("Görsel sayıları:", {s: len(RECORDS[s]) for s in SPLITS},
      "| arka plan (nesnesiz) görseller:", {s: sum(1 for r in RECORDS[s] if not r["cls"]) for s in SPLITS})
print("\nSınıf başına NESNE sayısı:");  display(inst.assign(toplam=inst.sum(axis=1)))
print("Sınıf başına GÖRSEL sayısı:");  display(imgs_with)

tr = inst["train"][inst["train"] > 0]
print(f"\nTrain dengesizlik oranı (max/min) = {tr.max()/tr.min():.1f}x  | en nadir: {tr.idxmin()} ({tr.min()}), en sık: {tr.idxmax()} ({tr.max()})")
for s in ("valid", "test"):
    empty = [n for n in NAMES if inst.loc[n, s] == 0]
    if empty:
        print(f"⚠️ {s} kümesinde HİÇ örneği olmayan sınıflar: {empty} -> bu sınıflar için metrik hesaplanamaz.")

fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
inst[list(SPLITS)].plot.bar(ax=ax[0], title="Sınıf başına nesne sayısı"); ax[0].set_yscale("log"); ax[0].set_ylabel("log ölçek")
(inst.div(inst.sum(axis=0), axis=1) * 100)[list(SPLITS)].plot.bar(ax=ax[1], title="Küme içi sınıf oranı (%) — dağılımlar benzer olmalı")
allb = np.array([b for r in RECORDS["train"] for b in r["boxes"]]) if any(r["boxes"] for r in RECORDS["train"]) else np.zeros((0, 4))
if len(allb):
    ax[2].hist(np.sqrt(allb[:, 2] * allb[:, 3]), bins=50); ax[2].set_title("Kutu boyutu √(w·h) (normalize) — küçük nesne yoğunluğu")
    print(f"Kutuların %{(np.sqrt(allb[:,2]*allb[:,3]) < 0.05).mean()*100:.1f}'i görüntü kenarının %5'inden küçük (küçük nesne; IMGSZ'yi düşürmeyin).")
plt.tight_layout(); plt.show()

## 5) Veri sızıntısı (data leakage) denetimi

Train/valid/test arasında şu yollarla sızıntı aranır:
1. **Roboflow kaynak kimliği** — `ad_jpg.rf.<hash>.jpg` biçiminde, aynı kaynak görselin augment edilmiş kopyaları farklı kümelere düşmüş olabilir.
2. **MD5** — birebir aynı dosya.
3. **dHash (görsel benzerlik)** — yeniden boyutlandırılmış/hafif değiştirilmiş neredeyse-aynı görseller, video kareleri.

**Politika:** `EXCLUDE_LEAKS=False` iken sızıntı yalnızca raporlanır, veri silinmez. `True` yapılırsa: `test` hiç değiştirilmez. `valid` içinde test ile çakışanlar valid'den, `train` içinde valid/test ile çakışanlar train'den **dışlanır** (liste dosyası ile; orijinal dosyalara dokunulmaz).

In [ ]:
def md5_file(p: Path) -> str:
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def dhash64(p: Path):
    try:
        with Image.open(p) as im:
            im = im.convert("L").resize((9, 8), Image.LANCZOS)
        a = np.asarray(im, dtype=np.int16)
        bits = (a[:, 1:] > a[:, :-1]).flatten()
        return np.frombuffer(np.packbits(bits).tobytes(), dtype=np.uint64)[0]
    except Exception:
        return None

def roboflow_source(p: Path):
    return p.name.split(".rf.")[0] if ".rf." in p.name else None

POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming_to_many(h, arr):
    x = np.bitwise_xor(arr, h)
    return POPCOUNT[x.view(np.uint8).reshape(-1, 8)].sum(axis=1)

FP = {}   # küme -> liste[(path, md5, dhash, kaynak)]
for s in SPLITS:
    FP[s] = [(p, md5_file(p), dhash64(p) if RUN_NEAR_DUP else None, roboflow_source(p))
             for p in tqdm(IMAGES[s], desc=f"parmak izi/{s}")]

def find_leaks(src, refs):
    # src kümesindeki hangi görseller refs kümelerindeki görsellerle çakışıyor?
    ref_md5, ref_src, ref_hash = set(), set(), []
    for r in refs:
        for p, m, h, sid in FP[r]:
            ref_md5.add(m)
            if sid: ref_src.add(sid)
            if h is not None: ref_hash.append(h)
    ref_arr = np.array(ref_hash, dtype=np.uint64)
    leaks = {}
    for p, m, h, sid in FP[src]:
        if m in ref_md5:
            leaks[p] = "birebir aynı dosya (MD5)"
        elif sid and sid in ref_src:
            leaks[p] = "aynı Roboflow kaynak görseli"
        elif h is not None and len(ref_arr) and hamming_to_many(h, ref_arr).min() <= NEAR_DUP_HAMMING:
            leaks[p] = "neredeyse aynı görsel (dHash)"
    return leaks

LEAKS = {"train": find_leaks("train", ["valid", "test"]),
         "valid": find_leaks("valid", ["test"]),
         "test":  {}}
EXCLUDE = {s: (set(LEAKS[s]) if EXCLUDE_LEAKS else set()) for s in SPLITS}

for s in ("train", "valid"):
    print(f"{s}: {len(LEAKS[s])} sızıntılı görsel tespit edildi ({len(RECORDS[s])} görselin %{100*len(LEAKS[s])/max(1,len(RECORDS[s])):.1f}'i) -> " + ("DIŞLANACAK" if EXCLUDE_LEAKS else "dışlanmayacak (EXCLUDE_LEAKS=False)"))
    for p, why in list(LEAKS[s].items())[:5]:
        print(f"    - {p.name}  [{why}]")
if not any(LEAKS.values()):
    print("✅ Kümeler arası sızıntı bulunmadı.")

# Son doğrulama: temizlenmiş kümeler arasında birebir kesişim kalmadığını kanıtla
clean_md5 = {s: {m for p, m, h, sid in FP[s] if p not in EXCLUDE[s]} for s in SPLITS}
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    common = clean_md5[a] & clean_md5[b]
    if EXCLUDE_LEAKS:
        assert not common, f"{a} ve {b} arasında hâlâ aynı dosya var!"
    elif common:
        print(f"⚠️ {a} ve {b} arasında {len(common)} birebir aynı dosya var -> test/valid skorları iyimser olabilir.")
if EXCLUDE_LEAKS:
    print("✅ Temizlenmiş kümeler MD5 düzeyinde ayrık.")

## 6) Sınıf dengesizliği: Repeat-Factor Sampling + sınıf ağırlıkları

**Repeat-Factor Sampling (LVIS):** sınıf *c* için `r_c = max(1, √(t / f_c))` (`f_c` = *c* sınıfını içeren train görsellerinin oranı). Bir görselin tekrar sayısı, içindeki sınıfların en büyük `r_c` değeridir (`RFS_MAX_REPEAT` ile sınırlı). Tekrarlar her epoch'ta farklı augmentation (mosaic/mixup/HSV...) aldığı için birebir kopya gibi ezberletmez.

**Sınıf ağırlıkları:** RFS sonrası *gerçek eğitim dağılımından* hesaplanır (çift sayım olmaması için), yumuşak (karekök) ve kırpılmıştır; 8. bölümdeki özel focal-BCE kaybında kullanılır.

In [ ]:
clean_train = [r for r in RECORDS["train"] if r["img"] not in EXCLUDE["train"]]
N_TR = len(clean_train)
assert N_TR > 0, "Sızıntı temizliği sonrası train boş kaldı!"

img_per_class = Counter(c for r in clean_train for c in set(r["cls"]))
rc = {c: max(1.0, math.sqrt(RFS_T / (img_per_class[c] / N_TR))) for c in img_per_class}

rng = np.random.default_rng(SEED)
REPEATS = {}
for r in clean_train:
    k = 1
    if USE_RFS and r["cls"]:
        ri = min(max(rc[c] for c in set(r["cls"])), RFS_MAX_REPEAT)
        k = int(math.floor(ri)); k += int(rng.random() < (ri - k))
    REPEATS[r["img"]] = max(1, k)

inst_after = Counter()
for r in clean_train:
    for c in r["cls"]:
        inst_after[c] += REPEATS[r["img"]]

cnt = np.array([inst_after.get(c, 0) for c in range(NC)], dtype=float)
present = cnt > 0
CLASS_W = np.ones(NC)
ref = np.median(cnt[present])
CLASS_W[present] = (ref / cnt[present]) ** CLASS_W_POWER
CLASS_W = np.clip(CLASS_W, *CLASS_W_CLIP)
CLASS_W[present] = CLASS_W[present] / CLASS_W[present].mean()   # ortalama 1: toplam kayıp ölçeği korunur

tbl = pd.DataFrame({
    "train_nesne_önce": inst["train"].values,
    "train_nesne_RFS_sonrası": cnt.astype(int),
    "sınıf_ağırlığı": CLASS_W.round(3),
}, index=NAMES)
print(f"Train görseli (temiz): {N_TR} -> RFS sonrası epoch başına {sum(REPEATS.values())} örnek")
display(tbl)
if present.any():
    print(f"Dengesizlik oranı: önce {tbl['train_nesne_önce'][tbl['train_nesne_önce']>0].max()/max(1,tbl['train_nesne_önce'][tbl['train_nesne_önce']>0].min()):.1f}x -> sonra {cnt[present].max()/cnt[present].min():.1f}x")

## 7) `data.yaml` üretimi ve doğrulama

İki YAML yazılır (kök dizine):

- **`data_train.yaml`** → yalnızca `train` + `val`. **`test` anahtarı yoktur**; eğitim, early stopping, checkpoint seçimi ve eşik seçimi test verisini görme şansına sahip değildir.
- **`data.yaml`** → `train` + `val` + `test`; yalnızca nihai bağımsız değerlendirmede kullanılır.

Sızıntı/RFS nedeniyle liste gerekiyorsa, `./` ile başlayan **göreli** yollu `.txt` dosyaları kök dizine yazılır (Ultralytics bunları `.txt` konumuna göre çözer; böylece liste içeriğinde Türkçe karakter/kesme işareti bulunmaz).

In [ ]:
def write_list(fname: str, items):
    # items: [(img_path, repeat), ...]  -> kök dizine göreli './klasör/images/x.jpg' satırları
    lines = []
    for p, k in items:
        rel = p.relative_to(ROOT).as_posix()
        lines.extend([f"./{rel}"] * k)
    (ROOT / fname).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return fname

USE_TRAIN_LIST = USE_RFS or bool(EXCLUDE["train"])
train_entry = write_list("train_balanced.txt", [(r["img"], REPEATS[r["img"]]) for r in clean_train]) if USE_TRAIN_LIST \
              else f"{SPLIT_DIR['train']}/images"

if EXCLUDE["valid"]:
    valid_entry = write_list("valid_clean.txt", [(r["img"], 1) for r in RECORDS["valid"] if r["img"] not in EXCLUDE["valid"]])
else:
    valid_entry = f"{SPLIT_DIR['valid']}/images"
test_entry = f"{SPLIT_DIR['test']}/images"

base = {"path": ROOT.as_posix(), "nc": NC, "names": {i: n for i, n in enumerate(NAMES)}}
data_train = {**base, "train": train_entry, "val": valid_entry}                 # test YOK
data_eval  = {**base, "train": train_entry, "val": valid_entry, "test": test_entry}

DATA_TRAIN_YAML = ROOT / "data_train.yaml"
DATA_EVAL_YAML  = ROOT / "data.yaml"

if DATA_EVAL_YAML.exists() and not (ROOT / "data.yaml.bak").exists():
    shutil.copy2(DATA_EVAL_YAML, ROOT / "data.yaml.bak")   # orijinal Roboflow yaml yedeği

for path, d in ((DATA_TRAIN_YAML, data_train), (DATA_EVAL_YAML, data_eval)):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(d, f, allow_unicode=True, sort_keys=False)

# ---- doğrulama
for path in (DATA_TRAIN_YAML, DATA_EVAL_YAML):
    d = yaml.safe_load(path.read_text(encoding="utf-8"))
    r = Path(d["path"])
    assert r.exists(), f"path yok: {r}"
    for k in ("train", "val", "test"):
        if k in d:
            assert (r / d[k]).exists(), f"{path.name}: '{k}' yolu bulunamadı: {r / d[k]}"
    assert d["nc"] == len(d["names"]) == NC
assert "test" not in yaml.safe_load(DATA_TRAIN_YAML.read_text(encoding="utf-8")), "Eğitim yaml'ında test olmamalı!"

try:
    from ultralytics.data.utils import check_det_dataset
    for path in (DATA_TRAIN_YAML, DATA_EVAL_YAML):
        chk = check_det_dataset(str(path), autodownload=False)
        print(f"✅ Ultralytics doğrulaması geçti: {path.name} | nc={chk['nc']}")
except Exception as e:
    raise RuntimeError(f"Ultralytics veri seti doğrulaması başarısız: {e}")

print("\n--- data_train.yaml ---"); print(DATA_TRAIN_YAML.read_text(encoding="utf-8"))
print("--- data.yaml (nihai test için) ---"); print(DATA_EVAL_YAML.read_text(encoding="utf-8"))

## 8) Model + özel kayıp (sınıf ağırlıklı focal-BCE)

**Neden özel kayıp?** Ultralytics'in sınıflandırma kaybı standart BCE'dir ve güncel sürümlerde `fl_gamma`/sınıf ağırlığı parametresi sunmaz. Burada `v8DetectionLoss` içindeki `bce` elemanı, **sınıf ağırlıklı + (Quality-)focal** bir modülle değiştirilir:

`L = BCE(p, y) · |y − σ(p)|^γ · w_c`

Yama, eğitim başlarken bir callback ile uygulanır; kayıp nesnesi (tek başlı veya `one2many/one2one` çift başlı mimariler dahil) otomatik taranır. Yama uygulanamazsa eğitim standart kayıpla **uyarı verilerek** devam eder.

> Bu bir *sezgisel* ayardır: `valid` mAP50-95'i ile karşılaştırmak için `USE_WEIGHTED_FOCAL=False` ile bir baseline koşusu yapmanız önerilir.

In [ ]:
class WeightedFocalBCE(nn.Module):
    def __init__(self, class_weights, gamma: float = 1.0):
        super().__init__()
        self.register_buffer("w", torch.as_tensor(class_weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, pred, target):
        loss = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
        if self.gamma > 0:
            loss = loss * (target - pred.sigmoid()).abs().pow(self.gamma)
        return loss * self.w.to(device=pred.device, dtype=loss.dtype)      # son eksen = sınıf

def patch_bce(crit, new_bce) -> int:
    # kayıp nesnesindeki (ve iç içe alt kayıpların) 'bce' modülünü değiştirir; kaç yerin yamalandığını döner
    patched, seen, stack = 0, set(), [crit]
    while stack:
        o = stack.pop()
        if id(o) in seen:
            continue
        seen.add(id(o))
        if isinstance(getattr(o, "bce", None), nn.Module):
            if hasattr(o, "nc"):
                assert o.nc == len(CLASS_W), f"nc uyuşmazlığı: kayıp={o.nc}, ağırlık={len(CLASS_W)}"
            o.bce = new_bce; patched += 1
        for v in getattr(o, "__dict__", {}).values():
            if hasattr(v, "bce") or "Loss" in type(v).__name__:
                stack.append(v)
    return patched

def apply_custom_loss(trainer):
    m = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
    crit = m.init_criterion()
    dev = next(m.parameters()).device
    n = patch_bce(crit, WeightedFocalBCE(CLASS_W, FOCAL_GAMMA).to(dev))
    m.criterion = crit
    if n:
        print(f"✅ Özel kayıp aktif: {n} sınıflandırma kaybı yamalandı (gamma={FOCAL_GAMMA}, ağırlıklar={np.round(CLASS_W, 2).tolist()})")
    else:
        print("⚠️ Özel kayıp yamalanamadı; standart Ultralytics kaybı ile devam ediliyor (RFS + augmentation yine aktif).")

model, used_weights = None, None
if RESUME and RESUME_WEIGHTS:
    model, used_weights = YOLO(str(RESUME_WEIGHTS)), str(RESUME_WEIGHTS)
else:
    for cand in MODEL_CANDIDATES:
        try:
            model, used_weights = YOLO(cand), cand
            break
        except Exception as e:
            print(f"'{cand}' yüklenemedi ({type(e).__name__}: {str(e)[:120]}) -> sıradaki aday deneniyor")
if model is None:
    raise RuntimeError("Hiçbir model ağırlığı yüklenemedi. İnternet bağlantısını / Ultralytics sürümünü kontrol edin.")
print("Kullanılan ön-eğitimli model:", used_weights)

if USE_WEIGHTED_FOCAL:
    model.add_callback("on_train_start", apply_custom_loss)

## 9) Eğitim

**Overfitting önlemleri**
- **Early stopping:** Ultralytics `patience`, her epoch'ta `valid` üzerinde hesaplanan *fitness = 0.1·mAP50 + 0.9·mAP50-95* değerini izler (yani mAP50-95'e bağlıdır). `best.pt` de bu ölçüte göre seçilir. **Test kullanılmaz.**
- **AdamW + weight decay (1e-3)**, **cosine LR** + warmup, yerleşik **EMA**.
- **Güçlü augmentation** (mosaic, mixup, HSV, perspective, flip, ölçek/öteleme) ve **`close_mosaic`** (son epoch'larda mosaic kapanır).
- *Dropout notu:* Ultralytics'te `dropout` parametresi yalnızca **sınıflandırma** başlığında etkilidir; nesne tespitinde etkisizdir, bu yüzden kullanılmaz. Eşdeğer düzenlileştirme için weight decay, augmentation, EMA, RFS tavanı ve isteğe bağlı `FREEZE` kullanılır.
- `flipud=0`: kamera sahnesinde gökyüzü/zemin ayrımı var; dikey çevirme gerçekçi değil.
- `copy_paste` yalnızca segmentasyon etiketlerinde etkili olduğundan kullanılmaz.

In [ ]:
if CACHE == "auto":
    try:
        import psutil
        n_samples = sum(REPEATS.values()) + len(IMAGES["valid"])          # RFS tekrarları ayrı önbelleklenir
        need = n_samples * IMGSZ * IMGSZ * 3 * 0.6                        # kaba tahmin (bayt)
        avail = psutil.virtual_memory().available
        CACHE = "ram" if need < 0.5 * avail else False
        print(f"Önbellek kararı: tahmini {need/1e9:.1f} GB / boş RAM {avail/1e9:.1f} GB -> cache={CACHE}")
    except Exception as e:
        CACHE = False; print("RAM tahmini yapılamadı, cache kapalı:", e)

train_args = dict(
    data=str(DATA_TRAIN_YAML),             # test içermeyen yaml
    epochs=EPOCHS, patience=PATIENCE,
    imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=RUN_NAME, exist_ok=False,
    seed=SEED, deterministic=True,
    # --- optimizasyon / düzenlileştirme
    optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, cos_lr=True,
    momentum=0.937, weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS,
    # --- kayıp kazançları (focal kaybı toplam kaybı küçülttüğü için cls biraz yüksek)
    box=7.5, cls=1.0 if USE_WEIGHTED_FOCAL else 0.5, dfl=1.5,
    # --- augmentation
    mosaic=1.0, mixup=0.1, close_mosaic=CLOSE_MOSAIC,
    fliplr=0.5, flipud=0.0,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0005,
    # --- diğer
    amp=(DEVICE == 0), cache=CACHE, plots=True, val=True, save=True, pretrained=True, verbose=True,
)
if FREEZE > 0:
    train_args["freeze"] = FREEZE
if RESUME and RESUME_WEIGHTS:
    train_args = {"resume": True}

print(json.dumps({k: (str(v) if isinstance(v, Path) else v) for k, v in train_args.items()}, indent=2, ensure_ascii=False))
results = model.train(**train_args)

RUN_DIR = Path(model.trainer.save_dir)
BEST    = RUN_DIR / "weights" / "best.pt"
print("\nEğitim klasörü:", RUN_DIR, "| best.pt mevcut:", BEST.exists())

## 10) Eğitim eğrileri ve overfitting teşhisi

In [ ]:
res = pd.read_csv(RUN_DIR / "results.csv")
res.columns = [c.strip() for c in res.columns]
res["fitness"] = 0.1 * res["metrics/mAP50(B)"] + 0.9 * res["metrics/mAP50-95(B)"]
best_ep = int(res["fitness"].idxmax())

groups = [
    ("Box loss", ["train/box_loss", "val/box_loss"]),
    ("Cls loss", ["train/cls_loss", "val/cls_loss"]),
    ("DFL loss", ["train/dfl_loss", "val/dfl_loss"]),
    ("mAP", ["metrics/mAP50(B)", "metrics/mAP50-95(B)"]),
    ("Precision / Recall", ["metrics/precision(B)", "metrics/recall(B)"]),
    ("Fitness (early-stopping ölçütü)", ["fitness"]),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (title, cols) in zip(axes.ravel(), groups):
    for c in cols:
        if c in res:
            ax.plot(res.index + 1, res[c], label=c.split("/")[-1])
    ax.axvline(best_ep + 1, color="gray", ls="--", lw=1, label=f"en iyi epoch={best_ep+1}")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

last, best = res.iloc[-1], res.iloc[best_ep]
print(f"Toplam epoch: {len(res)} | en iyi epoch: {best_ep+1} | valid mAP50-95: {best['metrics/mAP50-95(B)']:.4f} | mAP50: {best['metrics/mAP50(B)']:.4f}")
for k in ("box_loss", "cls_loss"):
    if f"train/{k}" in res and f"val/{k}" in res:
        gap = best[f"val/{k}"] - best[f"train/{k}"]
        print(f"En iyi epoch'ta {k}: train={best[f'train/{k}']:.3f} val={best[f'val/{k}']:.3f} (fark={gap:+.3f})")
if len(res) - (best_ep + 1) >= PATIENCE:
    print(f"⏹ Early stopping tetiklendi: {PATIENCE} epoch boyunca iyileşme olmadı.")
print("İpucu: val loss artarken mAP düşüyorsa overfitting vardır -> IMGSZ/model boyutunu küçültün, FREEZE>0 deneyin, WEIGHT_DECAY'i artırın.")

## 11) Bağımsız değerlendirme

Sıra **önemli** (sızıntıyı önlemek için):
1. **Çalışma eşiği (confidence)** `valid` kümesindeki F1 eğrisinden seçilir — test verisine bakılmaz.
2. `model.val(split='test', conf=0.001)` → standart mAP (tüm eşikler üzerinden).
3. `model.val(split='test', conf=<valid'den seçilen eşik>)` → bu eşikteki P / R / F1 ve Confusion Matrix.

In [ ]:
best_model = YOLO(str(BEST))

def run_val(split: str, conf: float, name: str, iou: float = 0.6):
    return best_model.val(data=str(DATA_EVAL_YAML), split=split, imgsz=IMGSZ, batch=VAL_BATCH,
                          conf=conf, iou=iou, device=DEVICE, workers=WORKERS, plots=True,
                          project=str(RUNS_DIR), name=name, exist_ok=True, verbose=False)

# ---- 1) eşik seçimi: yalnızca VALID
m_val = run_val("val", 0.001, f"{RUN_NAME}_valid_threshold")
BEST_CONF = 0.25
try:
    f1c, px = np.asarray(m_val.box.f1_curve), np.asarray(m_val.box.px)
    mean_f1 = f1c.mean(axis=0)
    smooth = np.convolve(mean_f1, np.ones(11) / 11, mode="same")
    BEST_CONF = float(np.clip(px[int(np.argmax(smooth))], 0.05, 0.90))
    plt.figure(figsize=(7, 4)); plt.plot(px, mean_f1, label="valid ort. F1"); plt.axvline(BEST_CONF, color="r", ls="--", label=f"seçilen eşik={BEST_CONF:.2f}")
    plt.xlabel("confidence"); plt.ylabel("F1"); plt.title("Eşik seçimi (VALID kümesi)"); plt.grid(alpha=.3); plt.legend(); plt.show()
except Exception as e:
    print("F1 eğrisi okunamadı, varsayılan eşik 0.25 kullanılıyor:", e)
print(f"Seçilen çalışma eşiği (valid): conf = {BEST_CONF:.3f}")

In [ ]:
# ---- 2) TEST: standart mAP (conf=0.001)
m_test_map = run_val("test", 0.001, f"{RUN_NAME}_test_map")
# ---- 3) TEST: valid'den seçilen eşikte P/R/F1 + confusion matrix
m_test_op  = run_val("test", BEST_CONF, f"{RUN_NAME}_test_op")

print("=== TEST (bağımsız) — tüm sınıflar ===")
print(f"mAP50      : {m_test_map.box.map50:.4f}")
print(f"mAP50-95   : {m_test_map.box.map:.4f}")
mp, mr = float(m_test_op.box.mp), float(m_test_op.box.mr)
print(f"Precision  : {mp:.4f}   (conf={BEST_CONF:.2f})")
print(f"Recall     : {mr:.4f}   (conf={BEST_CONF:.2f})")
print(f"F1 (makro) : {2*mp*mr/(mp+mr+1e-12):.4f}")

def per_class_table(m_op, m_map):
    rows = []
    b_op, b_map = m_op.box, m_map.box
    ap_lookup = {int(ci): b_map.class_result(i) for i, ci in enumerate(b_map.ap_class_index)}
    for i, ci in enumerate(b_op.ap_class_index):
        p, r, _, _ = b_op.class_result(i)
        _, _, ap50, ap = ap_lookup.get(int(ci), (np.nan,) * 4)
        rows.append({"sınıf": NAMES[int(ci)], "P": p, "R": r, "F1": 2*p*r/(p+r+1e-12), "AP50": ap50, "AP50-95": ap,
                     "test_nesne": int(inst.loc[NAMES[int(ci)], "test"])})
    return pd.DataFrame(rows).set_index("sınıf")

PER_CLASS = per_class_table(m_test_op, m_test_map)
display(PER_CLASS.round(4))

missing = [n for n in NAMES if n not in PER_CLASS.index]
if missing:
    print("⚠️ Test kümesinde örneği olmadığı için değerlendirilemeyen sınıflar:", missing)

ax = PER_CLASS[["P", "R", "F1", "AP50", "AP50-95"]].plot.bar(figsize=(12, 4.5), title="Test: sınıf bazında metrikler")
ax.set_ylim(0, 1.05); ax.grid(alpha=.3); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
# ---- Confusion Matrix (satır = tahmin, sütun = gerçek; son satır/sütun = arka plan)
cm = np.asarray(m_test_op.confusion_matrix.matrix, dtype=float)
labels = NAMES + ["background"]

def plot_cm(mat, title, fmt):
    fig, ax = plt.subplots(figsize=(max(7, 0.9 * len(labels)), max(6, 0.8 * len(labels))))
    im = ax.imshow(mat, cmap="Blues"); plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Gerçek sınıf"); ax.set_ylabel("Tahmin"); ax.set_title(title)
    thr = mat.max() / 2 if mat.max() > 0 else 1
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center", color="white" if mat[i, j] > thr else "black", fontsize=8)
    plt.tight_layout(); plt.show()

plot_cm(cm, f"Confusion Matrix — TEST (sayı, conf={BEST_CONF:.2f}, IoU=0.5)", ".0f")
plot_cm(cm / (cm.sum(axis=0, keepdims=True) + 1e-9), "Confusion Matrix — TEST (gerçek sınıfa göre normalize)", ".2f")

fn = cm[-1, :-1].sum(); fp = cm[:-1, -1].sum()
print(f"Kaçırılan nesne (FN, arka plan olarak tahmin) = {int(fn)} | Yanlış alarm (FP, arka plana karşı) = {int(fp)}")

# Ultralytics'in kendi PR / F1 / confusion grafikleri
def show_saved(run_dir, names):
    for n in names:
        p = Path(run_dir) / n
        if p.exists():
            plt.figure(figsize=(8, 6)); plt.imshow(Image.open(p)); plt.axis("off"); plt.title(n); plt.show()
show_saved(m_test_map.save_dir, ["PR_curve.png", "F1_curve.png"])

## 12) Çıkarım (inference) ve tespit kutularını çizme

In [ ]:
def detect_and_draw(weights_or_model, image_paths, conf=None, iou=0.6, imgsz=None, cols=2, save_dir=None):
    # Verilen görsellerde tespit yapar, kutuları çizer, matplotlib ile gösterir.
    # Dönüş: ultralytics Results listesi.
    mdl = YOLO(str(weights_or_model)) if isinstance(weights_or_model, (str, Path)) else weights_or_model
    conf = BEST_CONF if conf is None else conf
    imgsz = IMGSZ if imgsz is None else imgsz
    image_paths = [Path(p) for p in image_paths]
    results = mdl.predict([str(p) for p in image_paths], conf=conf, iou=iou, imgsz=imgsz,
                          device=DEVICE, verbose=False)
    rows = math.ceil(len(results) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    if save_dir:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
    for ax, p, r in zip(axes.ravel(), image_paths, results):
        plotted = r.plot(line_width=2)                       # BGR numpy
        ax.imshow(plotted[..., ::-1])                        # -> RGB
        summary = Counter(mdl.names[int(c)] for c in r.boxes.cls.tolist())
        ax.set_title(f"{p.name[:40]}\ntahmin: {dict(summary) if summary else 'nesne yok'}", fontsize=10)
        if save_dir:
            Image.fromarray(plotted[..., ::-1]).save(Path(save_dir) / f"pred_{p.stem}.jpg")
    plt.tight_layout(); plt.show()
    return results

# Test kümesinden rastgele (etiketli) örnekler
rnd = random.Random(SEED)
labeled_test = [r["img"] for r in RECORDS["test"] if r["cls"]]
sample = rnd.sample(labeled_test, k=min(6, len(labeled_test)))
_ = detect_and_draw(best_model, sample, save_dir=RUN_DIR / "sample_predictions")
print("Örnek tahminler kaydedildi:", RUN_DIR / "sample_predictions")

## 13) Dışa aktarma ve özet raporu

In [ ]:
onnx_path = None
if EXPORT_ONNX:
    try:
        onnx_path = best_model.export(format="onnx", imgsz=IMGSZ, simplify=True)
        print("ONNX:", onnx_path)
    except Exception as e:
        print("ONNX export başarısız (eğitim sonucu etkilenmez):", e)

summary = {
    "model": used_weights,
    "ultralytics": ultralytics.__version__,
    "weights_best": str(BEST),
    "onnx": str(onnx_path) if onnx_path else None,
    "classes": NAMES,
    "imgsz": IMGSZ,
    "operating_conf_selected_on_valid": BEST_CONF,
    "leakage_excluded": {"train": len(EXCLUDE["train"]), "valid": len(EXCLUDE["valid"])},
    "rfs": {"enabled": USE_RFS, "t": RFS_T, "max_repeat": RFS_MAX_REPEAT},
    "weighted_focal": {"enabled": USE_WEIGHTED_FOCAL, "gamma": FOCAL_GAMMA, "class_weights": CLASS_W.round(4).tolist()},
    "test": {
        "mAP50": float(m_test_map.box.map50), "mAP50-95": float(m_test_map.box.map),
        "precision": mp, "recall": mr, "f1": float(2*mp*mr/(mp+mr+1e-12)),
        "per_class": PER_CLASS.round(4).reset_index().to_dict(orient="records"),
    },
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Özet kaydedildi:", RUN_DIR / "summary.json")
print(json.dumps({k: summary[k] for k in ("model", "operating_conf_selected_on_valid", "leakage_excluded")}, indent=2, ensure_ascii=False))

## Notlar ve sonraki adımlar

- **Test kümesi tam izole:** eğitim yaml'ında `test` yok; eşik `valid`'den seçildi; `best.pt` valid fitness'ına göre seçildi. Test yalnızca bu son hücrelerde kullanıldı. Test sonuçlarına bakıp hiperparametre değiştirirseniz o noktadan sonra test artık bağımsız değildir.
- **Dengesizlik hâlâ sorunluysa:** `RFS_T`'yi artırın (0.15–0.3), `CLASS_W_POWER`'ı 0.75'e çıkarın, nadir sınıflar için daha fazla gerçek veri toplayın. Etiket kalitesi (özellikle nadir sınıflar) çoğu zaman en büyük kaldıraçtır.
- **Overfitting sürüyorsa:** `MODEL_CANDIDATES`'ı `s` boyutuna çevirin, `FREEZE=10`, `WEIGHT_DECAY=2e-3`, `RFS_MAX_REPEAT=2`.
- **Baseline karşılaştırması:** `USE_RFS=False, USE_WEIGHTED_FOCAL=False` ile aynı seed'de bir koşu yapıp valid mAP50-95 ve nadir sınıf recall'unu karşılaştırın.
- **Disk:** `data.yaml.bak` orijinal Roboflow yaml'ınızın yedeğidir; `train_balanced.txt` / `valid_clean.txt` kök dizine yazılan liste dosyalarıdır (orijinal görsellere dokunulmaz).